In [ ]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import glob
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import numpy as np
np.random.seed(0)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torch.optim.lr_scheduler import MultiStepLR 

import lpips
from degradations.degradation import create_degradation_pipeline

from project_folder.best_model1 import Vision
from project_folder.best_model2 import Vision2

def seed_everything(seed=0):
    import random
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
        
CONFIG = {
    'total_iterations': 30000,   
    'batch_size': 32,
    'learning_rate': 1e-3,
    'patch_size': 256,
    'val_interval': 1000,
    'num_workers': 8,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'plot_path': 'loss_graph/wavelet_val_psnr_curve.png',
    'lr_milestones': [100000, 150000, 175000, 187500, 195000, 198000], 
    'lr_gamma': 0.5
}

os.makedirs(os.path.dirname(CONFIG['plot_path']), exist_ok=True)

class RestorationDataset(Dataset):
    def __init__(self, image_paths, patch_size=256):
        self.patch_size = patch_size
        self.image_paths = image_paths
        
        if len(self.image_paths) == 0:
            raise ValueError("Empty image list provided to dataset")
            
        self.degradation_transform = create_degradation_pipeline()
        self.to_tensor = transforms.ToTensor()
        self.crop = transforms.RandomCrop(patch_size)
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        gt_img = Image.open(img_path).convert('RGB')
        
        if gt_img.size[0] < self.patch_size or gt_img.size[1] < self.patch_size:
            pad_transform = transforms.Pad((0, 0, max(0, self.patch_size - gt_img.size[0]), 
                                                  max(0, self.patch_size - gt_img.size[1])))
            gt_img = pad_transform(gt_img)
            
        gt_patch = self.crop(gt_img)
        gt_patch_np = np.array(gt_patch)

        augmented = self.degradation_transform(image=gt_patch_np)
        lq_patch_np = augmented['image']
        
        lq_patch = self.to_tensor(lq_patch_np)
        gt_patch = self.to_tensor(gt_patch)
            
        return lq_patch, gt_patch


class BenchmarkDataset(Dataset):
    def __init__(self, root_dir):
        self.clean_dir = os.path.join(root_dir, 'clean')
        self.degraded_dir = os.path.join(root_dir, 'degraded')
        
        self.filenames = sorted([f for f in os.listdir(self.clean_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        gt_img = Image.open(os.path.join(self.clean_dir, fname)).convert('RGB')
        lq_img = Image.open(os.path.join(self.degraded_dir, fname)).convert('RGB')
        
        return self.to_tensor(lq_img), self.to_tensor(gt_img)

def create_val_benchmark(val_source_paths, output_dir, num_patches=10000, patch_size=256):
    if os.path.exists(output_dir):
        print(f"--> Валидационный бенчмарк '{output_dir}' обнаружен. Генерация не требуется.")
        return 
        
    clean_dir = os.path.join(output_dir, 'clean')
    degraded_dir = os.path.join(output_dir, 'degraded')

    os.makedirs(clean_dir, exist_ok=True)
    os.makedirs(degraded_dir, exist_ok=True)
    
    transform = create_degradation_pipeline()
    print(f"--> Генерируем {num_patches} валидационных патчей в {output_dir}...")
    
    pbar = tqdm(total=num_patches)
    count = 0
    while count < num_patches:
        img_path = random.choice(val_source_paths)
        
        try:
            img_pil = Image.open(img_path).convert('RGB')
        except:
            continue
            
        w, h = img_pil.size
        if w < patch_size or h < patch_size:
            continue
            
        left = random.randint(0, w - patch_size)
        top = random.randint(0, h - patch_size)
        clean_patch_pil = img_pil.crop((left, top, left + patch_size, top + patch_size))
        
        clean_patch_np = np.array(clean_patch_pil)
        
        degraded_patch_np = transform(image=clean_patch_np)['image']
        
        fname = f"val_{count:05d}.png"
        clean_patch_pil.save(os.path.join(clean_dir, fname))
        Image.fromarray(degraded_patch_np).save(os.path.join(degraded_dir, fname))
        
        count += 1
        pbar.update(1)
    pbar.close()

def calculate_psnr_batch(output, target):
    """Считает PSNR для батча тензоров"""
    mse = torch.mean((output - target) ** 2, dim=[1, 2, 3])
    psnr = 10 * torch.log10(1.0 / (mse + 1e-10))
    return torch.mean(psnr).item()

def save_metrics_plot(iterations, metrics_dict, path):
    """
    Строит 4 отдельных графика на одном изображении (Grid 2x2).
    """
    plt.figure(figsize=(14, 10))
    
    plt.subplot(2, 2, 1)
    plt.plot(iterations, metrics_dict['PSNR'], color='tab:blue')
    plt.xlabel('Iterations')
    plt.ylabel('dB')
    plt.title('PSNR (Higher is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 2)
    plt.plot(iterations, metrics_dict['SSIM'], color='tab:green')
    plt.xlabel('Iterations')
    plt.title('SSIM (Higher is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 3)
    plt.plot(iterations, metrics_dict['LPIPS'], color='tab:red')
    plt.xlabel('Iterations')
    plt.title('LPIPS (Lower is better)')
    plt.grid(True)
    
    plt.subplot(2, 2, 4)
    plt.plot(iterations, metrics_dict['DISTS'], color='tab:orange')
    plt.xlabel('Iterations')
    plt.title('DISTS (Lower is better)')
    plt.grid(True)
        
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()
    
class ComplexLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.2, device="cuda"):
        super(ComplexLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        
        self.l1_loss = nn.L1Loss()
        
        if self.beta > 0:
            self.lpips = lpips.LPIPS(net='alex').to(device)
            self.lpips.eval()
        else:
            self.lpips = None

    def forward(self, pred, target):
        loss_l1 = self.l1_loss(pred, target)
        total_loss = self.alpha * loss_l1
        
        if self.lpips is not None:
            pred_lpips = pred.clamp(0.0, 1.0) * 2.0 - 1.0
            target_lpips = target.clamp(0.0, 1.0) * 2.0 - 1.0
            
            loss_lpips_val = torch.mean(self.lpips(pred_lpips, target_lpips))
            total_loss += self.beta * loss_lpips_val
        
        return total_loss

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision(kernel_size_first=9, dwt_level=2, groups_11=8, heads_52=8, exp_52=2.66, exp_129=2.66, last_kernel_size=5, last_rcab_ca=4, net_width=6).to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(kernel_size_first=9, dwt_level=2, groups_11=8, heads_52=8, exp_52=2.66, exp_129=2.66, last_kernel_size=5, last_rcab_ca=4, net_width=6)")

In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision(kernel_size_first=9, dwt_level=2, groups_11=8, heads_52=8, exp_52=2.66, exp_129=2.66, last_kernel_size=5, last_rcab_ca=2, net_width=6).to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
                
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(kernel_size_first=9, dwt_level=2, groups_11=8, heads_52=8, exp_52=2.66, exp_129=2.66, last_kernel_size=5, last_rcab_ca=2, net_width=6)")

In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    

    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(vision2)")

In [ ]:
from project_folder.WaveletFusionNet import WaveletFusionNet

In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = WaveletFusionNet(
        in_channels=3,
        out_channels=3,
        base_width=12,
        depth=2,
        enc_blk_nums=2,
        dec_blk_nums=2,
        gfm_inter_ratio=1.5,
        ca_reduction=2
    ).to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(wavelet)")

In [ ]:
from layers import DWTLayer
from layers import IWTLayer
from layers import RCAB
from layers import RestormerBlock
import torch
import torch.nn as nn
import torch.nn.functional as F

class Vision2_rep(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.layer_128 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=9, stride=1, padding=4, dilation=1, groups=3)
        self.layer_128_1 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=7, stride=1, padding=3, dilation=1, groups=3)
        self.layer_128_2 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, stride=1, padding=2, dilation=1, groups=3)
        self.layer_128_3 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, stride=1, padding=1, dilation=1, groups=3)
        self.layer_128_4 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=1, stride=1, padding=0, dilation=1, groups=3)
        
        self.layer_3 = DWTLayer(wave='haar', level=2)
        self.layer_43 = nn.GroupNorm(num_groups=16, num_channels=48)
        self.layer_11 = nn.Conv2d(in_channels=48, out_channels=96, kernel_size=1, stride=1, padding=0, dilation=1, groups=8)
        self.layer_27 = nn.AdaptiveAvgPool2d((1, 1))
        self.layer_106 = nn.PReLU()
        
        self.layer_28 = nn.Conv2d(in_channels=96, out_channels=12, kernel_size=7, stride=2, padding=3, dilation=1, groups=2)
        self.layer_28_1 = nn.Conv2d(in_channels=96, out_channels=12, kernel_size=5, stride=2, padding=2, dilation=1, groups=2)
        self.layer_28_2 = nn.Conv2d(in_channels=96, out_channels=12, kernel_size=3, stride=2, padding=1, dilation=1, groups=2)
        self.layer_28_3 = nn.Conv2d(in_channels=96, out_channels=12, kernel_size=1, stride=2, padding=0, dilation=1, groups=2)
        
        self.layer_52 = RestormerBlock(dim=96, num_heads=8, ffn_expansion_factor=2.66, bias=False, LayerNorm_type='BiasFree')
        self.layer_116 = nn.LeakyReLU()
        self.layer_36 = RCAB(num_feat=96, ca_reduction=8, bias=True)
        self.layer_125 = nn.PReLU()
        self.layer_123 = nn.Mish()
        self.layer_30 = nn.Conv2d(in_channels=12, out_channels=96, kernel_size=1, stride=1, padding=0, dilation=1, groups=2)
        self.layer_131 = nn.GELU()
        self.layer_126 = nn.LeakyReLU()
        self.layer_115 = nn.PReLU()
        self.layer_127 = nn.GELU()
        self.layer_129 = RestormerBlock(dim=96, num_heads=1, ffn_expansion_factor=2.66, bias=True, LayerNorm_type='WithBias')
        self.layer_119 = nn.SiLU()
        self.layer_108 = nn.Conv2d(in_channels=96, out_channels=12, kernel_size=1, stride=2, padding=0, dilation=1, groups=2)
        self.layer_110 = nn.Conv2d(in_channels=12, out_channels=96, kernel_size=1, stride=1, padding=0, dilation=1, groups=1)
        self.layer_111 = nn.Sigmoid()
        self.layer_130 = nn.Mish()
        self.layer_6 = IWTLayer(wave='haar', level=2)
        self.layer_21 = RCAB(num_feat=9, ca_reduction=16, bias=True)
        
        self.layer_2 = nn.Conv2d(in_channels=9, out_channels=3, kernel_size=5, stride=1, padding=2, dilation=1, groups=1)
        self.layer_2_1 = nn.Conv2d(in_channels=9, out_channels=3, kernel_size=3, stride=1, padding=1, dilation=1, groups=1)
        self.layer_2_2 = nn.Conv2d(in_channels=9, out_channels=3, kernel_size=1, stride=1, padding=0, dilation=1, groups=1)
        

    def forward(self, x):
        x_0 = x
        x_128 = self.layer_128(x_0) + self.layer_128_1(x_0) + self.layer_128_2(x_0) + self.layer_128_3(x_0) + self.layer_128_4(x_0)
        x_3 = self.layer_3(x_128)
        x_43 = self.layer_43(x_3)
        x_11 = self.layer_11(x_43)
        x_27 = self.layer_27(x_11)
        x_106 = self.layer_106(x_11)
        x_28 = self.layer_28(x_27) + self.layer_28_1(x_27) + self.layer_28_2(x_27) + self.layer_28_3(x_27)
        x_52 = self.layer_52(x_106)
        x_116 = self.layer_116(x_28)
        x_36 = self.layer_36(x_52)
        x_125 = self.layer_125(x_116)
        x_123 = self.layer_123(x_125)
        x_30 = self.layer_30(x_123)
        x_131 = self.layer_131(x_30)
        x_126 = self.layer_126(x_131)
        x_115 = self.layer_115(x_126)
        x_127 = self.layer_127(x_115)
        x_129 = self.layer_129(x_127)
        x_119 = self.layer_119(x_129)
        x_108 = self.layer_108(x_119)
        x_110 = self.layer_110(x_108)
        x_111 = self.layer_111(x_110)
        x_130 = self.layer_130(x_111)
        x_112 = x_30 * x_130
        x_32 = x_36 * x_112
        x_6 = self.layer_6(x_32)
        x_7 = torch.cat([x_0, x_6], dim=1)
        x_21 = self.layer_21(x_7)
        x_2 = self.layer_2(x_21) + self.layer_2_1(x_21) + self.layer_2_2(x_21)
        return x_2


In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2_rep().to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                with open('best_config.txt', "a") as f:
                    f.write(f"{history}(vision2_rep)\n")
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(vision2_rep)\n")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_dim, 1),
            nn.SiLU(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)

class MBConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio, stride):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        
        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU()
            ])
        
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU()
        ])
        
        layers.append(SEBlock(hidden_dim, max(1, hidden_dim // 4)))
        
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        res = self.block(x)
        if self.use_residual:
            return x + res
        return res

class EfficientRouter(nn.Module):
    def __init__(self, num_experts=8):
        super().__init__()
        
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.SiLU()
        )
        
        self.blocks = nn.Sequential(
            MBConvBlock(in_channels=16, out_channels=24, expand_ratio=1, stride=2),
            
            MBConvBlock(in_channels=24, out_channels=40, expand_ratio=4, stride=2),
            MBConvBlock(in_channels=40, out_channels=40, expand_ratio=4, stride=1),
            
            MBConvBlock(in_channels=40, out_channels=80, expand_ratio=4, stride=2),
        )
        
        self.head = nn.Sequential(
            nn.Conv2d(80, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, num_experts)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        logits = self.head(x)
        return logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoEImageEnhancer(nn.Module):
    def __init__(self, num_experts=8):
        super().__init__()
        self.num_experts = num_experts
        
        self.experts = nn.ModuleList([Vision2() for _ in range(num_experts)])
        
        self.router = EfficientRouter(num_experts=num_experts) 

    def forward(self, x_full, x_small=None):
        batch_size = x_full.size(0)
        
        if x_small is None:
            x_small = F.interpolate(x_full, size=(64, 64), mode='bilinear', align_corners=False)
        
        router_logits = self.router(x_small)
        
        if self.training:
            router_probs = F.gumbel_softmax(router_logits, tau=1.0, hard=True, dim=1)
        else:
            indices = torch.argmax(router_logits, dim=1)
            router_probs = F.one_hot(indices, num_classes=self.num_experts).float()
            
        output = torch.zeros_like(x_full)
        
        expert_indices = torch.argmax(router_probs, dim=1)
        
        for i, expert in enumerate(self.experts):
            batch_mask = (expert_indices == i) 
            
            if batch_mask.any():
                sub_input = x_full[batch_mask]
                
                sub_output = expert(sub_input)
                
                output.index_put_((batch_mask,), sub_output)
        
        if self.training:
            selected_probs = torch.gather(router_probs, 1, expert_indices.unsqueeze(1)) 
            selected_probs = selected_probs.view(-1, 1, 1, 1)
            output = output * selected_probs
        
        return output, router_logits
        
        
class MoEAuxLoss(nn.Module):
    def __init__(self, num_experts=8, w_balance=0.1, w_entropy=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.w_balance = w_balance
        self.w_entropy = w_entropy

    def forward(self, router_logits):
        probs = F.softmax(router_logits, dim=1)
        
        entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1)
        entropy_loss = torch.mean(entropy)
        
        f = torch.mean(probs, dim=0)
        balance_loss = self.num_experts * torch.sum(f * f)
        
        aux_loss = (self.w_balance * balance_loss) + (self.w_entropy * entropy_loss)
        
        return aux_loss, {"bal": balance_loss.item(), "ent": entropy_loss.item()}

In [ ]:
if __name__ == "__main__":
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = MoEImageEnhancer().to(CONFIG['device'])
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=CONFIG['learning_rate']
    )
    
    criterion = ComplexLoss(device=CONFIG['device'])

    balancing_criterion = MoEAuxLoss(num_experts=8, w_balance=0.02, w_entropy=0.005).cuda()
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output, logits = model(lq)

            bal_loss, _ = balancing_criterion(logits)

            main_loss = criterion(output, gt)

            with open('bal_loss.txt', "a") as f:
                    f.write(f"{main_loss}\t{bal_loss}\n")
            
            loss = main_loss + bal_loss
            
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    expert_counts = torch.zeros(8, dtype=torch.long, device='cuda')
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output, router_logits = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)

                        indices = torch.argmax(router_logits, dim=1) # [Batch_Size]
            
                        batch_counts = torch.bincount(indices, minlength=8)
                        
                        expert_counts += batch_counts
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(expert_counts.cpu().tolist())
                with open('best_config.txt', "a") as f:
                    f.write(f"{history}(MoE)\n")
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}(MoE)\n")

# ТЕСТ ОПТИМИЗАТОРОВ

In [ ]:
from pytorch_optimizer import Lion, AdEMAMix, ADOPT, MARS, SPAM, EXAdam, AdaGC, EmoNavi, ScheduleFreeAdamW, ScheduleFreeRAdam, SOAP, Ranger, Ranger25

if __name__ == "__main__":
    for opt_name in ['AdamW', 'RAdam', 'NAdam', 'Adamax', 'AdEMAMix', 'Lion', 'ADOPT', 'MARS', 'SPAM', 'EXAdam', 'AdaGC', 'EmoNavi', 'ScheduleFreeAdamW', 'ScheduleFreeRAdam', 'SOAP', 'Ranger', 'Ranger25']:
        seed_everything()
        extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
        all_files = []
        for ext in extensions:
            found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
            all_files.extend(found_files)
            
        all_files.sort()
        
        random.seed(42)
        random.shuffle(all_files)
        
        split_idx = int(0.9 * len(all_files))
        train_files = all_files[:split_idx]
        val_files_source = all_files[split_idx:]
        
        BENCHMARK_DIR = "val_benchmark_fixed"
        create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)
    
        seed_everything(42)
        train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
        val_dataset = BenchmarkDataset(BENCHMARK_DIR)
        
        print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
        
        g = torch.Generator()
        g.manual_seed(42)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=CONFIG['batch_size'], 
            shuffle=True,
            num_workers=CONFIG['num_workers'],
            worker_init_fn=seed_worker,
            generator=g,
            pin_memory=True,
            drop_last=True,
            persistent_workers=False
        )
        
        val_loader = DataLoader(
            val_dataset, 
            batch_size=CONFIG['batch_size'], 
            shuffle=False,
            num_workers=CONFIG['num_workers'],
            pin_memory=True,
            persistent_workers=False
        )
        
        model = Vision2().to(CONFIG['device'])
        
        if opt_name == 'AdamW':
            optimizer = optim.AdamW(model.parameters())
            
        elif opt_name == 'RAdam':
            optimizer = optim.RAdam(model.parameters())
            
        elif opt_name == 'NAdam':
            optimizer = optim.NAdam(model.parameters())
            
        elif opt_name == 'Adamax':
            optimizer = optim.Adamax(model.parameters())

        elif opt_name == 'AdEMAMix':
            optimizer = AdEMAMix(model.parameters())

        elif opt_name == 'Lion':
            optimizer = Lion(model.parameters())

        elif opt_name == 'ADOPT':
            optimizer = ADOPT(model.parameters())

        elif opt_name == 'MARS':
            optimizer = MARS(model.parameters())

        elif opt_name == 'SPAM':
            optimizer = SPAM(model.parameters())

        elif opt_name == 'EXAdam':
            optimizer = EXAdam(model.parameters())

        elif opt_name == 'AdaGC':
            optimizer = AdaGC(model.parameters())

        elif opt_name == 'EmoNavi':
            optimizer = EmoNavi(model.parameters())

        elif opt_name == 'ScheduleFreeAdamW':
            optimizer = ScheduleFreeAdamW(model.parameters(), decoupling_c=200)

        elif opt_name == 'ScheduleFreeRAdam':
            optimizer = ScheduleFreeRAdam(model.parameters())

        elif opt_name == 'SOAP':
            optimizer = SOAP(model.parameters())

        elif opt_name == 'Ranger':
            optimizer = Ranger(model.parameters())

        elif opt_name == 'Ranger25':
            optimizer = Ranger25(model.parameters())
        
        criterion = ComplexLoss(device=CONFIG['device'])
        
        current_iter = 0
        
        history = {
            'LPIPS': []
        }
        iter_history = []
        
        model.train()
        
        while current_iter < CONFIG['total_iterations']:
            for i, (lq, gt) in enumerate(train_loader):
                if current_iter >= CONFIG['total_iterations']:
                    break
                
                lq = lq.to(CONFIG['device'])
                gt = gt.to(CONFIG['device'])
                
                optimizer.zero_grad()
                
                output = model(lq)
                
                loss = criterion(output, gt)
        
                loss.backward()
                
                optimizer.step()
                
                current_iter += 1 
        
                if current_iter % CONFIG['val_interval'] == 0:
                    model.eval()
                    
                    metrics_accum = {'LPIPS': 0.0}
    
                    criterion.eval() 
                    
                    with torch.no_grad():
                        for val_lq, val_gt in val_loader:
                            
                            val_lq = val_lq.to(CONFIG['device'])
                            val_gt = val_gt.to(CONFIG['device'])
                            
                            val_output = model(val_lq)
                            val_output = torch.clamp(val_output, 0.0, 1.0)
                            
                            val_output_lpips = val_output * 2.0 - 1.0
                            val_gt_lpips = val_gt * 2.0 - 1.0
                            
                            batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                            metrics_accum['LPIPS'] += batch_lpips
                    
                    denom = len(val_loader) 
                    
                    for key in history.keys():
                        avg_val = metrics_accum[key] / denom
                        history[key].append(avg_val)
                    
                    iter_history.append(current_iter)
                    print(history)
                    
                    model.train()
        with open('best_config.txt', "a") as f:
            f.write(f"{history}({opt_name})\n")

In [ ]:
from pytorch_optimizer import GrokFastAdamW
if __name__ == "__main__":
    opt_name = 'GrokFastAdamW'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = GrokFastAdamW(model.parameters())
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")

In [ ]:
from pytorch_optimizer import Adalite
if __name__ == "__main__":
    opt_name = 'Adalite'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = Adalite(model.parameters())
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")

In [ ]:
from pytorch_optimizer import ScheduleFreeWrapper
if __name__ == "__main__":
    opt_name = 'ScheduleFreeSOAP'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = ScheduleFreeWrapper(SOAP(model.parameters()))
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    optimizer.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                optimizer.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
                optimizer.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")

In [ ]:
if __name__ == "__main__":
    opt_name = 'ScheduleFreeNAdam'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = ScheduleFreeWrapper(optim.NAdam(model.parameters()))
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    optimizer.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                optimizer.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
                optimizer.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")

In [ ]:
from pytorch_optimizer.base.type import OPTIMIZER_INSTANCE_OR_CLASS
from pytorch_optimizer.base.type import State
from pytorch_optimizer.base.type import Closure
from pytorch_optimizer.base.type import ParamGroup
from pytorch_optimizer.base.type import Loss
from pytorch_optimizer.base.optimizer import BaseOptimizer
from collections import defaultdict

class ScheduleFreeWrapper(BaseOptimizer):
    def __init__(
        self,
        optimizer: OPTIMIZER_INSTANCE_OR_CLASS,
        momentum: float = 0.9,
        weight_decay: float = 0.0,
        r: float = 0.0,
        weight_lr_power: float = 2.0,
        maximize: bool = False,
        decoupling_c: float = 200.0,
        **kwargs,
    ):
        self.validate_range(momentum, 'momentum', 0.0, 1.0, '[)')
        self.validate_non_negative(weight_decay, 'weight_decay')
        self.validate_non_negative(decoupling_c, 'decoupling_c')

        self.momentum = momentum
        self.weight_decay = weight_decay
        self.r = r
        self.weight_lr_power = weight_lr_power
        self.train_mode: bool = False
        self.maximize = maximize
        self.decoupling_c = decoupling_c

        self.optimizer: Optimizer = self.load_optimizer(optimizer, **kwargs)

        self._optimizer_step_pre_hooks: Dict[int, Callable] = {}
        self._optimizer_step_post_hooks: Dict[int, Callable] = {}

        self.state: State = defaultdict(dict)
        self.defaults: Defaults = self.optimizer.defaults

    def __str__(self) -> str:
        return 'ScheduleFree'

    @property
    def param_groups(self):
        return self.optimizer.param_groups

    def __getstate__(self):
        return {'state': self.state, 'optimizer': self.optimizer}

    def add_param_group(self, param_group):
        return self.optimizer.add_param_group(param_group)

    def state_dict(self) -> State:
        return {'schedulefree_state': self.state, 'base_optimizer': self.optimizer.state_dict()}

    def load_state_dict(self, state: State) -> None:
        r"""Load state."""
        self.state = state['schedulefree_state']
        self.optimizer.load_state_dict(state['base_optimizer'])

    def zero_grad(self, set_to_none: bool = True) -> None:
        self.optimizer.zero_grad(set_to_none)

    @torch.no_grad()
    def eval(self):
        if not self.train_mode:
            return

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                if 'z' in state:
                    p.lerp_(end=state['z'], weight=1.0 - 1.0 / self.momentum)

        self.train_mode = False

    @torch.no_grad()
    def train(self):
        if self.train_mode:
            return

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                if 'z' in state:
                    p.lerp_(end=state['z'], weight=1.0 - self.momentum)

        self.train_mode = True

    def init_group(self, group: ParamGroup, **kwargs) -> None:
        if 'step' not in group:
            group['step'] = 0

        for p in group['params']:
            if p.grad is None:
                continue

            grad = p.grad
            if grad.is_sparse:
                raise NoSparseGradientError(str(self))

            state = self.state[p]

            if 'z' not in state:
                state['z'] = p.clone()

    @staticmethod
    def swap(x: torch.Tensor, y: torch.Tensor) -> None:
        x.view(torch.uint8).bitwise_xor_(y.view(torch.uint8))
        y.view(torch.uint8).bitwise_xor_(x.view(torch.uint8))
        x.view(torch.uint8).bitwise_xor_(y.view(torch.uint8))

    @torch.no_grad()
    def step(self, closure: Closure = None) -> Loss:
        if not self.train_mode:
            raise ValueError('optimizer was not in train mode when step is called. call .train() before training')

        loss: Loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            self.init_group(group)
            group['step'] += 1

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad

                self.maximize_gradient(grad, maximize=self.maximize)

                state = self.state[p]

                z = state['z']

                self.apply_weight_decay(
                    z,
                    grad=grad,
                    lr=group['lr'],
                    weight_decay=self.weight_decay,
                    weight_decouple=True,
                    fixed_decay=False,
                )

                self.apply_weight_decay(
                    p,
                    grad=grad,
                    lr=group['lr'],
                    weight_decay=self.weight_decay,
                    weight_decouple=True,
                    fixed_decay=False,
                    ratio=1.0 - self.momentum,
                )

                p.lerp_(end=z, weight=1.0 - 1.0 / self.momentum)

                self.swap(z, p)

        self.optimizer.step()

        for group in self.param_groups:
            lr: float = group['lr'] * group.get('d', 1.0)
            lr_max = group['lr_max'] = max(lr, group.get('lr_max', 0))

            weight: float = (group['step'] ** self.r) * (lr_max ** self.weight_lr_power)  # fmt: skip
            weight_sum = group['weight_sum'] = group.get('weight_sum', 0.0) + weight

            checkpoint: float = weight / weight_sum if weight_sum != 0.0 else 0.0

            if self.decoupling_c > 0:
                beta1 = group['betas'][0]
                checkpoint = min(1.0, checkpoint * (1.0 - beta1) * self.decoupling_c)

            for p in group['params']:
                if p.grad is None:
                    continue

                state = self.state[p]

                z = state['z']

                self.swap(z, p)

                p.lerp_(end=z, weight=checkpoint)

                p.lerp_(end=state['z'], weight=1.0 - self.momentum)

        return loss

In [ ]:
if __name__ == "__main__":
    opt_name = 'RefinedScheduleFreeSOAP'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = ScheduleFreeWrapper(SOAP(model.parameters()))
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    optimizer.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                optimizer.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
                optimizer.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")

In [ ]:
if __name__ == "__main__":
    opt_name = 'RefinedScheduleFreeNAdam'
    seed_everything()
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif')
    all_files = []
    for ext in extensions:
        found_files = sorted(glob.glob(os.path.join("Datasets", '**', ext), recursive=True))
        all_files.extend(found_files)
        
    all_files.sort()
    
    random.seed(42)
    random.shuffle(all_files)
    
    split_idx = int(0.9 * len(all_files))
    train_files = all_files[:split_idx]
    val_files_source = all_files[split_idx:]
    
    BENCHMARK_DIR = "val_benchmark_fixed"
    create_val_benchmark(val_files_source, BENCHMARK_DIR, num_patches=10000)

    seed_everything(42)
    train_dataset = RestorationDataset(train_files, patch_size=CONFIG['patch_size'])
    val_dataset = BenchmarkDataset(BENCHMARK_DIR)
    
    print(f"Train files: {len(train_files)}, Benchmark patches: {len(val_dataset)}")
    
    g = torch.Generator()
    g.manual_seed(42)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
        drop_last=True,
        persistent_workers=False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True,
        persistent_workers=False
    )
    
    model = Vision2().to(CONFIG['device'])
    
    optimizer = ScheduleFreeWrapper(torch.optim.NAdam(model.parameters()))
    
    criterion = ComplexLoss(device=CONFIG['device'])
    
    current_iter = 0
    
    history = {
        'LPIPS': []
    }
    iter_history = []
    
    model.train()
    optimizer.train()
    
    while current_iter < CONFIG['total_iterations']:
        for i, (lq, gt) in enumerate(train_loader):
            if current_iter >= CONFIG['total_iterations']:
                break
            
            lq = lq.to(CONFIG['device'])
            gt = gt.to(CONFIG['device'])
            
            optimizer.zero_grad()
            
            output = model(lq)
            
            loss = criterion(output, gt)
    
            loss.backward()
            
            optimizer.step()
            
            current_iter += 1 
    
            if current_iter % CONFIG['val_interval'] == 0:
                model.eval()
                optimizer.eval()
                
                metrics_accum = {'LPIPS': 0.0}

                criterion.eval() 
                
                with torch.no_grad():
                    for val_lq, val_gt in val_loader:
                        
                        val_lq = val_lq.to(CONFIG['device'])
                        val_gt = val_gt.to(CONFIG['device'])
                        
                        val_output = model(val_lq)
                        val_output = torch.clamp(val_output, 0.0, 1.0)
                        
                        val_output_lpips = val_output * 2.0 - 1.0
                        val_gt_lpips = val_gt * 2.0 - 1.0
                        
                        batch_lpips = criterion.lpips(val_output_lpips, val_gt_lpips).mean().item()
                        metrics_accum['LPIPS'] += batch_lpips
                
                denom = len(val_loader) 
                
                for key in history.keys():
                    avg_val = metrics_accum[key] / denom
                    history[key].append(avg_val)
                
                iter_history.append(current_iter)
                print(history)
                
                model.train()
                optimizer.train()
    with open('best_config.txt', "a") as f:
        f.write(f"{history}({opt_name})\n")